## Cell A

In [1]:
# CELL A1 | Setup: import, cari root repo, baca config.yaml
import copy
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

FPS = 25


def find_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "main.py").is_file():
            return p
    raise FileNotFoundError("main.py tidak ditemukan dari folder kerja notebook.")


ROOT = find_root()
OUT = ROOT / "outputs" / "scenarios"
OUT.mkdir(parents=True, exist_ok=True)
base_cfg = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
print("ROOT:", ROOT)
print("control_min_s di config:", base_cfg["possession_v2"]["control"]["control_min_s"], "(harus 1.0)")

ROOT: C:\Users\nesha\VSCODE_ALL\more-than-a-hack-mbg
control_min_s di config: 1.0 (harus 1.0)


In [2]:
# CELL A2 | Fungsi bantu: set_path, run, durations
def set_path(cfg, dotted, value):
    keys = dotted.split(".")
    node = cfg
    for k in keys[:-1]:
        node = node[k]
    if keys[-1] not in node:
        raise KeyError(f"kunci tidak ada di config: {dotted}")
    node[keys[-1]] = value


def run(name, cfg):
    cp, op = OUT / f"{name}.yaml", OUT / f"{name}.csv"
    cp.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    r = subprocess.run([sys.executable, "main.py", "--config", str(cp), "--output", str(op)],
                       cwd=ROOT, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-500:], r.stderr[-1500:])
        raise RuntimeError(f"main.py gagal untuk skenario '{name}'")
    shape = next((l.split(":")[-1].strip() for l in r.stdout.splitlines()
                  if l.startswith("team-shape")), "-")
    lab = pd.read_csv(op)["label"].astype(str).str.strip().str.upper().to_numpy()
    return lab, shape


def durations(lab):
    out, prev, n = [], None, 0
    for x in lab:
        if x in ("H", "A") and x == prev:
            n += 1
            continue
        if prev is not None:
            out.append(n)
        prev, n = (x, 1) if x in ("H", "A") else (None, 0)
    if prev is not None:
        out.append(n)
    return np.array(out) / FPS

In [3]:
# CELL A3 | Daftar skenario (tempat satu-satunya untuk menambah atau mengubah skenario)
SCENARIOS = {
    "base": {"why": "config.yaml saat ini", "set": {}},
    "shape_clip_calibrated": {
        "why": "Keyakinan shape model tidak boleh melebihi akurasi held-out-nya: ln(0.85/0.15) = 1.7",
        "set": {"possession_v2.shape.clip": 1.7},
    },
    "shape_missing_weight_half": {
        "why": "Agreement 85% diukur saat bola terlihat; saat bola hilang bobot penuh bisa terlalu percaya diri",
        "set": {"possession_v2.evidence.shape_when_ball_missing": 0.5},
    },
    "ablation_no_shape": {
        "why": "Dokumen: seberapa jauh label bergantung pada shape model",
        "set": {"possession_v2.evidence.shape_when_ball_visible": 0.0,
                "possession_v2.evidence.shape_when_ball_missing": 0.0},
    },
    "ablation_no_tactical": {
        "why": "Dokumen: kontribusi Smart Tagging",
        "set": {"possession_v2.evidence.tactical_when_ball_visible": 0.0,
                "possession_v2.evidence.tactical_when_ball_missing": 0.0},
    },
    "ablation_no_restart_prior": {
        "why": "Dokumen: kontribusi tim yang melakukan restart",
        "set": {"possession_v2.restart.restart_team_weight": 0.0},
    },
    "own_radius_1_0": {
        "why": "Sensitivitas: 'di bola' = dalam satu langkah (lebih ketat)",
        "set": {"possession_v2.control.own_radius_m": 1.0},
    },
    "own_radius_2_0": {
        "why": "Sensitivitas (lebih longgar)",
        "set": {"possession_v2.control.own_radius_m": 2.0},
    },
    "team_switch_penalty_10": {
        "why": "Sensitivitas di control_min_s yang baru",
        "set": {"possession_v2.decoder.team_switch_penalty": 10.0},
    },
    "team_switch_penalty_20": {
        "why": "Sensitivitas di control_min_s yang baru",
        "set": {"possession_v2.decoder.team_switch_penalty": 20.0},
    },
}

In [4]:
# CELL A4 | Jalankan semua skenario, tampilkan tabel perbandingan (variabel: df)
ONLY = None   # contoh: ["base", "ablation_no_shape"]; None = semua skenario

rows, base_lab = [], None
for name, spec in SCENARIOS.items():
    if ONLY and name not in ONLY and name != "base":
        continue
    cfg = copy.deepcopy(base_cfg)
    for dotted, value in spec["set"].items():
        set_path(cfg, dotted, value)
    lab, shape = run(name, cfg)
    if base_lab is None:
        base_lab = lab
    d = durations(lab)
    graded = base_lab != "X"
    live = (lab == "H") | (lab == "A")
    rows.append({
        "skenario": name,
        "poss": len(d),
        "rata_s": round(float(d.mean()), 1),
        "<1s": int((d < 1).sum()),
        "<2s": int((d < 2).sum()),
        "D%": round(100 * float((lab == "D").sum() / max((lab != "X").sum(), 1)), 1),
        "H%live": round(100 * float((lab == "H").sum() / max(live.sum(), 1)), 1),
        "berubah%": round(100 * float((lab[graded] != base_lab[graded]).mean()), 2),
        "shape_cocok": shape,
    })
    print("selesai:", name, flush=True)

df = pd.DataFrame(rows)
df

selesai: base
selesai: shape_clip_calibrated
selesai: shape_missing_weight_half
selesai: ablation_no_shape
selesai: ablation_no_tactical
selesai: ablation_no_restart_prior
selesai: own_radius_1_0
selesai: own_radius_2_0
selesai: team_switch_penalty_10
selesai: team_switch_penalty_20


,skenario,poss,rata_s,<1s,<2s,D%,H%live,berubah%,shape_cocok
0,base,293,13.5,7,36,37.9,64.1,0.00,"86.8%, 85.6%"
1,shape_clip_calibrated,296,13.3,5,36,37.9,63.7,0.26,"86.8%, 85.6%"
2,shape_missing_weight_half,292,13.5,6,36,37.9,64.5,0.67,"86.8%, 85.6%"
3,ablation_no_shape,285,13.8,3,38,37.9,64.9,5.62,"86.8%, 85.6%"
4,ablation_no_tactical,314,12.6,7,33,37.9,63.1,2.79,"86.8%, 85.6%"
5,ablation_no_restart_prior,292,13.5,7,35,37.9,64.1,0.02,"86.8%, 85.6%"
6,own_radius_1_0,252,15.7,7,21,37.9,64.1,2.35,"89.6%, 89.0%"
7,own_radius_2_0,329,12.0,6,52,37.9,63.6,1.48,"85.2%, 83.3%"
8,team_switch_penalty_10,328,12.0,15,61,37.9,64.0,0.47,"86.8%, 85.6%"
9,team_switch_penalty_20,280,14.1,5,31,37.9,63.9,0.24,"86.8%, 85.6%"


# CELL B

In [5]:
# CELL B1 | Parameter dan sampel acak kejadian (di luar window publik)
PRED = ROOT / "outputs" / "submission.csv"
SEED = 7
N_SWITCH = 20     # kejadian ganti tim
N_DEAD = 10       # per jenis (bola mati mulai, bola mati selesai)
PUBLIC = (11185, 18685)   # window GT publik: dikecualikan

lab = pd.read_csv(PRED)["label"].astype(str).str.strip().str.upper().to_numpy()
rng = np.random.default_rng(SEED)

events = {"ganti_tim": [], "bola_mati_mulai": [], "bola_mati_selesai": []}
for t in range(1, len(lab)):
    if PUBLIC[0] <= t < PUBLIC[1]:
        continue
    a, b = lab[t - 1], lab[t]
    if a in ("H", "A") and b in ("H", "A") and a != b:
        events["ganti_tim"].append(t)
    elif a in ("H", "A") and b == "D":
        events["bola_mati_mulai"].append(t)
    elif a == "D" and b in ("H", "A"):
        events["bola_mati_selesai"].append(t)

sizes = {"ganti_tim": N_SWITCH, "bola_mati_mulai": N_DEAD, "bola_mati_selesai": N_DEAD}
picked = []
for kind, frames in events.items():
    k = min(sizes[kind], len(frames))
    for f in rng.choice(frames, size=k, replace=False):
        picked.append((kind, int(f)))
picked = [picked[i] for i in rng.permutation(len(picked))]
print({k: len(v) for k, v in events.items()}, "kejadian tersedia di luar window")
print(len(picked), "kejadian disampel")

{'ganti_tim': 157, 'bola_mati_mulai': 123, 'bola_mati_selesai': 121} kejadian tersedia di luar window
40 kejadian disampel


In [6]:
# CELL B2 | Simpan lembar pemeriksa (tanpa prediksi) dan kunci prediksi (terpisah)
def hms(frame):
    s = frame / FPS
    return f"{int(s // 3600)}:{int(s % 3600 // 60):02d}:{s % 60:05.2f}"


sheet, key = [], []
for i, (kind, f) in enumerate(picked, 1):
    sheet.append({
        "id": i,
        "tonton_dari": hms(max(0, f - 5 * FPS)),
        "tonton_sampai": hms(f + 5 * FPS),
        "pemegang_awal_jendela": "",    # H = Barcelona, A = Athletic, D = bola mati
        "pemegang_akhir_jendela": "",
        "waktu_perubahan": "",          # jam:menit:detik di pemutar video; kosong kalau tidak ada perubahan
        "catatan": "",
        "pemeriksa": "",
    })
    key.append({"id": i, "jenis": kind, "frame": f, "waktu_prediksi": hms(f),
                "sebelum": lab[f - 1], "sesudah": lab[f]})

SPOT = ROOT / "outputs" / "spotcheck"
SPOT.mkdir(parents=True, exist_ok=True)
pd.DataFrame(sheet).to_csv(SPOT / "lembar_cek.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(key).to_csv(SPOT / "kunci_prediksi.csv", index=False, encoding="utf-8-sig")
print("Lembar :", SPOT / "lembar_cek.csv")
print("Kunci  :", SPOT / "kunci_prediksi.csv", "(jangan dibagikan ke pemeriksa dulu)")

Lembar : C:\Users\nesha\VSCODE_ALL\more-than-a-hack-mbg\outputs\spotcheck\lembar_cek.csv
Kunci  : C:\Users\nesha\VSCODE_ALL\more-than-a-hack-mbg\outputs\spotcheck\kunci_prediksi.csv (jangan dibagikan ke pemeriksa dulu)


# Cell G

In [4]:
# CELL G1 | Setup: parameter, root repo, daftar kejadian (hanya id dan frame, tanpa prediksi)
import io
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

FPS = 25
WINDOW_S = 8                  # panjang tiap GIF (detik)
OFFSET_RANGE_S = (2.0, 6.0)   # kejadian jatuh 2 sampai 6 detik setelah awal GIF (diacak per id)
STEP = 2                      # ambil tiap 2 frame (0,08 detik)
DURATION_MS = 160             # lama tampil per frame di GIF -> sekitar 0,5x kecepatan asli
FLIP_Y = False                # ubah True kalau sisi atas dan bawah terbalik dibanding video
RENDER_IDS = [1, 2, 3]        # uji 3 kejadian dulu; nanti ganti None untuk semua


def find_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "main.py").is_file():
            return p
    raise FileNotFoundError("main.py tidak ditemukan dari folder kerja notebook.")


ROOT = find_root()
sys.path.insert(0, str(ROOT))
SPOT = ROOT / "outputs" / "spotcheck"
GIF_DIR = SPOT / "gif"
GIF_DIR.mkdir(parents=True, exist_ok=True)

key = pd.read_csv(SPOT / "kunci_prediksi.csv", usecols=["id", "frame"])
windows = {}
for r in key.itertuples():
    off = random.Random(1000 + int(r.id)).uniform(*OFFSET_RANGE_S)
    start = max(0, int(round(r.frame - off * FPS)))
    windows[int(r.id)] = (start, min(199318, start + WINDOW_S * FPS))

print("ROOT:", ROOT)
print(len(windows), "kejadian | contoh jendela id 1:", windows[1])

ROOT: C:\Users\nesha\VSCODE_ALL\more-than-a-hack-mbg
40 kejadian | contoh jendela id 1: (50133, 50333)


In [5]:
# CELL G2 | Baca metadata (slot -> tim) dan HANYA frame yang dibutuhkan dari file tracking
import games
from kloppy import metrica

paths = games.asset_paths(source="tactical")
ds = metrica.load_tracking_epts(meta_data=paths["atd_meta"], raw_data=paths["atd_raw"], limit=5)
teams = ds.metadata.teams
slot_team = np.array([0] * len(teams[0].players) + [1] * len(teams[1].players))   # 0 = H (Barcelona), 1 = A
PITCH_L, PITCH_W = games.GAMES[games.GAME]["pitch"]
KICKOFF = 11185

need = {KICKOFF}
for s, e in windows.values():
    need.update(range(s, e + 1))
max_i = max(need)


def parse_line(line):
    fr, pl, bl = line.rstrip("\r\n").split(":")
    arr = np.array([[float(v) for v in s.split(",")] for s in pl.split(";")], dtype=np.float32)
    ball = np.array([float(v) for v in bl.split(",")], dtype=np.float32)
    return int(fr), arr, ball


DATA = {}
with open(paths["atd_raw"], "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i in need:
            fr, arr, ball = parse_line(line)
            if fr != i + 1:
                raise ValueError(f"baris {i} berisi frame_id {fr}, harusnya {i + 1}")
            DATA[i] = (arr, ball)
        if i >= max_i:
            break

arr, _ = DATA[KICKOFF]
ok = np.isfinite(arr[:, :2]).all(axis=1)
hx = float(np.mean(arr[ok & (slot_team == 0), 0])) * PITCH_L
ax_ = float(np.mean(arr[ok & (slot_team == 1), 0])) * PITCH_L
print(len(DATA), "frame dimuat | slot:", len(slot_team), "| lapangan:", PITCH_L, "x", PITCH_W)
print(f"kickoff: rata-rata x home={hx:.1f} m, away={ax_:.1f} m, selisih={abs(hx - ax_):.1f} m")
print("HARUS: selisih > sekitar 15 m (dua tim di sisi lapangan berlawanan). Kalau tidak, STOP dan kabari aku.")

7634 frame dimuat | slot: 46 | lapangan: 105.3 x 68.0
kickoff: rata-rata x home=40.2 m, away=54.4 m, selisih=14.2 m
HARUS: selisih > sekitar 15 m (dua tim di sisi lapangan berlawanan). Kalau tidak, STOP dan kabari aku.


In [6]:
# CELL G3 | Gambar lapangan dan fungsi render_frame(frame, win_start) -> gambar PIL (tanpa pyplot dan widget)
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.patches import Circle, Rectangle

fig = Figure(figsize=(8, 5.3), dpi=70)
FigureCanvasAgg(fig)
ax = fig.add_subplot(111)
ax.set_xlim(-3, PITCH_L + 3)
ax.set_ylim(-3, PITCH_W + 3)
ax.set_aspect("equal")
ax.axis("off")
ax.add_patch(Rectangle((0, 0), PITCH_L, PITCH_W, fill=False, lw=1.5, ec="#555"))
ax.plot([PITCH_L / 2] * 2, [0, PITCH_W], color="#555", lw=1)
ax.add_patch(Circle((PITCH_L / 2, PITCH_W / 2), 9.15, fill=False, lw=1, ec="#555"))
for x0, sign in ((0, 1), (PITCH_L, -1)):
    for depth, width in ((16.5, 40.32), (5.5, 18.32)):
        ax.add_patch(Rectangle((x0 if sign > 0 else x0 - depth, (PITCH_W - width) / 2),
                               depth, width, fill=False, lw=1, ec="#555"))
    ax.plot([x0 + sign * 11], [PITCH_W / 2], "o", color="#555", ms=2)

home_sc = ax.scatter([], [], s=110, c="#2457c5", edgecolors="white", zorder=3)
away_sc = ax.scatter([], [], s=110, c="#d62728", edgecolors="white", zorder=3)
ball_sc = ax.scatter([], [], s=130, c="white", edgecolors="black", linewidths=1.5, zorder=5)
trail, = ax.plot([], [], "-", color="black", alpha=0.4, lw=2, zorder=4)
ax.text(1, -2.4, "biru = Barcelona (H)    merah = Athletic (A)", fontsize=8, color="#444")
title = ax.set_title("", fontsize=10)


def hms(frame):
    s = frame / FPS
    return f"{int(s // 3600)}:{int(s % 3600 // 60):02d}:{s % 60:05.2f}"


def xy_m(a):
    x = a[..., 0] * PITCH_L
    y = a[..., 1] * PITCH_W
    if FLIP_Y:
        y = PITCH_W - y
    return np.stack([x, y], axis=-1)


def render_frame(frame, win_start=None):
    arr, ball = DATA[frame]
    pos = arr[:, :2]
    valid = np.isfinite(pos).all(axis=1)
    home_sc.set_offsets(xy_m(pos[valid & (slot_team == 0)]))
    away_sc.set_offsets(xy_m(pos[valid & (slot_team == 1)]))
    ball_ok = bool(np.isfinite(ball[:2]).all())
    ball_sc.set_offsets(xy_m(ball[None, :2]) if ball_ok else np.empty((0, 2)))
    pts = [DATA[k][1][:2] for k in range(max(0, frame - 12), frame + 1)
           if k in DATA and np.isfinite(DATA[k][1][:2]).all()]
    if len(pts) >= 2:
        t = xy_m(np.array(pts))
        trail.set_data(t[:, 0], t[:, 1])
    else:
        trail.set_data([], [])
    rel = "" if win_start is None else f"   +{(frame - win_start) / FPS:.2f}s"
    status = "" if ball_ok else "   BOLA TIDAK TERDETEKSI"
    title.set_text(f"frame {frame}   video {hms(frame)}{rel}{status}")
    title.set_color("black" if ball_ok else "#c0392b")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=70)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

In [7]:
# CELL G4 | Simpan gambar kickoff (cek orientasi) lalu GIF tiap kejadian ke outputs/spotcheck/gif/
import time

render_frame(KICKOFF).save(SPOT / "kickoff_cek.png")
print("tersimpan:", SPOT / "kickoff_cek.png", "(buka dan cek: dua tim di sisi berlawanan)")

ids = sorted(windows) if RENDER_IDS is None else RENDER_IDS
for i in ids:
    s, e = windows[i]
    t0 = time.time()
    imgs = [render_frame(f, s).convert("P", palette=Image.ADAPTIVE, colors=64)
            for f in range(s, e + 1, STEP)]
    path = GIF_DIR / f"kejadian_{i:02d}.gif"
    imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=DURATION_MS,
                 loop=0, optimize=False, disposal=2)
    print(f"kejadian {i:2d} | frame {s}-{e} | video {hms(s)} sampai {hms(e)} | "
          f"{len(imgs)} frame | {time.time() - t0:.0f} detik | {path.name}", flush=True)

tersimpan: C:\Users\nesha\VSCODE_ALL\more-than-a-hack-mbg\outputs\spotcheck\kickoff_cek.png (buka dan cek: dua tim di sisi berlawanan)
kejadian  1 | frame 50133-50333 | video 0:33:25.32 sampai 0:33:33.32 | 101 frame | 4 detik | kejadian_01.gif
kejadian  2 | frame 82803-83003 | video 0:55:12.12 sampai 0:55:20.12 | 101 frame | 4 detik | kejadian_02.gif
kejadian  3 | frame 40610-40810 | video 0:27:04.40 sampai 0:27:12.40 | 101 frame | 4 detik | kejadian_03.gif


In [8]:
# CELL G5 | Simpan hasil satu kejadian ke lembar_cek_terisi.csv (format sama, bisa dibaca CELL B3)
RESULTS_PATH = SPOT / "lembar_cek_terisi.csv"
COLS = ["id", "pemegang_awal_jendela", "pemegang_akhir_jendela", "waktu_perubahan",
        "frame_perubahan", "catatan", "pemeriksa"]


def simpan(event_id, awal, akhir, frame_perubahan=None, catatan="", pemeriksa=""):
    """awal/akhir: 'H', 'A', 'D', atau '' kalau tidak yakin. frame_perubahan: nomor frame, atau None.
    Contoh: simpan(1, 'A', 'A', catatan='...', pemeriksa='Shaf')"""
    if awal not in ("H", "A", "D", "") or akhir not in ("H", "A", "D", ""):
        raise ValueError("awal/akhir harus 'H', 'A', 'D', atau ''")
    s, e = windows[event_id]
    if frame_perubahan is not None and not (s <= frame_perubahan <= e):
        raise ValueError(f"frame_perubahan harus di antara {s} dan {e}")
    row = {
        "id": str(event_id),
        "pemegang_awal_jendela": awal,
        "pemegang_akhir_jendela": akhir,
        "waktu_perubahan": "" if frame_perubahan is None else f"{frame_perubahan / FPS:.2f}",
        "frame_perubahan": "" if frame_perubahan is None else str(frame_perubahan),
        "catatan": catatan,
        "pemeriksa": pemeriksa,
    }
    if RESULTS_PATH.exists():
        old = pd.read_csv(RESULTS_PATH, dtype=str).fillna("")
        old = old[old["id"].astype(int) != event_id]
    else:
        old = pd.DataFrame(columns=COLS)
    out = pd.concat([old, pd.DataFrame([row])], ignore_index=True)
    out = out.sort_values("id", key=lambda c: c.astype(int))
    out.to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")
    print(f"kejadian {event_id} tersimpan | total {len(out)}/{len(windows)}")

In [9]:
simpan(1, "A", "D", pemeriksa="Shaf",
       catatan="merah menendang ke gawang biru, bola keluar (out); tidak ada biru yang berhasil merebut; bola hilang di tracking karena sudah keluar")
simpan(2, "D", "A", pemeriksa="Shaf",
       catatan="bola mati dulu, keeper merah letakkan bola di tanah lalu menendang, diterima merah; bola hilang saat dipegang keeper, terdeteksi lagi saat diletakkan")
simpan(3, "H", "A", pemeriksa="Shaf",
       catatan="offside biru, possession pindah ke merah; kemungkinan ada henti pendek di antaranya")

kejadian 1 tersimpan | total 3/40
kejadian 2 tersimpan | total 3/40
kejadian 3 tersimpan | total 3/40


In [10]:
# CELL G6 | Bandingkan catatanmu dengan prediksi model (hanya untuk id yang sudah kamu simpan)
def bandingkan():
    if not RESULTS_PATH.exists():
        print("belum ada yang disimpan")
        return
    mine = pd.read_csv(RESULTS_PATH, dtype=str).fillna("")
    mine["id"] = mine["id"].astype(int)
    k = pd.read_csv(SPOT / "kunci_prediksi.csv")
    m = k.merge(mine, on="id", how="inner")
    rows = []
    for r in m.itertuples():
        awal, akhir = r.pemegang_awal_jendela, r.pemegang_akhir_jendela
        fp = int(r.frame_perubahan) if r.frame_perubahan else None
        rows.append({
            "id": r.id,
            "jenis": r.jenis,
            "model": f"{r.sebelum} -> {r.sesudah}",
            "kamu": f"{awal or '?'} -> {akhir or '?'}",
            "cocok_awal": None if not awal else awal == r.sebelum,
            "cocok_akhir": None if not akhir else akhir == r.sesudah,
            "err_s": None if fp is None else round((r.frame - fp) / FPS, 2),
        })
    print("Catatan: 'awal' kamu = awal GIF (2-6 detik sebelum kejadian model), 'model sebelum' = label tepat sebelum kejadian.")
    print("Kalau ada perubahan lain di antaranya, keduanya bisa beda tanpa berarti model salah.")
    return pd.DataFrame(rows)


bandingkan()

Catatan: 'awal' kamu = awal GIF (2-6 detik sebelum kejadian model), 'model sebelum' = label tepat sebelum kejadian.
Kalau ada perubahan lain di antaranya, keduanya bisa beda tanpa berarti model salah.


,id,jenis,model,kamu,cocok_awal,cocok_akhir,err_s
0,1,bola_mati_mulai,A -> D,A -> D,True,True,None
1,2,bola_mati_selesai,D -> A,D -> A,True,True,None
2,3,ganti_tim,H -> A,H -> A,True,True,None


In [11]:
# CELL G7 | Diagnosis window publik: bola hilang berkorelasi dengan label GT apa?
GT_PATH = ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"
PRED_PATH = ROOT / "outputs" / "submission.csv"
LO, HI = 11185, 18685


def _read(p):
    return pd.read_csv(p)["label"].astype(str).str.strip().str.upper().to_numpy()


gt, pr = _read(GT_PATH), _read(PRED_PATH)

known = np.zeros(HI, dtype=bool)
with open(paths["atd_raw"], "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= HI:
            break
        if i < LO:
            continue
        b = line.rstrip("\r\n").rsplit(":", 1)[-1].split(",")
        known[i] = np.isfinite(float(b[0])) and np.isfinite(float(b[1]))

g, p, k = gt[LO:HI], pr[LO:HI], known[LO:HI]
print("bola terlihat per label GT (window publik):")
for lab in ("H", "A", "D"):
    m = g == lab
    print(f"  {lab}: {int(m.sum()):5d} frame | bola terlihat {k[m].mean():.0%}")


def runs_of(mask):
    edges = np.diff(np.r_[0, mask.astype(np.int8), 0])
    return list(zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)))


def comp(arr):
    vals, cnt = np.unique(arr, return_counts=True)
    order = np.argsort(-cnt)
    return " / ".join(f"{vals[j]} {cnt[j] / len(arr):.0%}" for j in order)


MIN_RUN = 2 * FPS
print("\nStretch bola hilang >= 2 detik (waktu = detik dari awal window):")
print(" mulai  durasi  GT                     prediksi")
tot, tot_d = 0, 0
for s, e in runs_of(~k):
    if e - s >= MIN_RUN:
        print(f"{s / FPS:6.1f} {(e - s) / FPS:7.1f}  {comp(g[s:e]):22} {comp(p[s:e])}")
        tot += e - s
        tot_d += int((g[s:e] == "D").sum())
if tot:
    print(f"\ntotal di stretch panjang: {tot / FPS:.0f} detik | di antaranya GT = D: {tot_d / tot:.0%}")

bola terlihat per label GT (window publik):
  H:  3454 frame | bola terlihat 100%
  A:  2577 frame | bola terlihat 85%
  D:  1469 frame | bola terlihat 66%

Stretch bola hilang >= 2 detik (waktu = detik dari awal window):
 mulai  durasi  GT                     prediksi
  65.8     5.9  D 86% / A 14%          H 59% / A 41%
 218.4     5.3  A 100%                 A 100%
 234.5     9.3  A 100%                 A 100%
 251.0    14.6  D 100%                 D 100%

total di stretch panjang: 35 detik | di antaranya GT = D: 56%


In [12]:
# CELL G8 | Seluruh match: stretch bola hilang >= 4 detik, dan dilabeli apa oleh model (tanpa GT)
import numpy as np
import pandas as pd

known_all = []
with open(paths["atd_raw"], "r", encoding="utf-8") as f:
    for line in f:
        b = line.rstrip("\r\n").rsplit(":", 1)[-1].split(",")
        known_all.append(np.isfinite(float(b[0])) and np.isfinite(float(b[1])))
known_all = np.array(known_all)

pred = pd.read_csv(ROOT / "outputs" / "submission.csv")["label"].astype(str).str.strip().str.upper().to_numpy()
graded = pred != "X"
missing = ~known_all & graded

edges = np.diff(np.r_[0, missing.astype(np.int8), 0])
runs_m = list(zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)))
long_runs = [(s, e) for s, e in runs_m if e - s >= 4 * FPS]

tot = sum(e - s for s, e in long_runs)
by_label = {l: int(sum((pred[s:e] == l).sum() for s, e in long_runs)) for l in ("H", "A", "D")}
print(f"bola hilang di frame dinilai: {missing.sum() / graded.sum():.0%}")
print(f"stretch hilang >= 4 detik: {len(long_runs)} | total {tot / FPS / 60:.1f} menit")
for l, c in by_label.items():
    print(f"  dilabeli {l}: {c / tot:.0%}")
lens = np.array([(e - s) / FPS for s, e in long_runs])
print(f"durasi stretch (detik): median {np.median(lens):.1f} | p90 {np.percentile(lens, 90):.1f} | maks {lens.max():.0f}")

bola hilang di frame dinilai: 41%
stretch hilang >= 4 detik: 133 | total 43.0 menit
  dilabeli H: 14%
  dilabeli A: 10%
  dilabeli D: 76%
durasi stretch (detik): median 11.6 | p90 38.7 | maks 221


# Cell H

In [18]:
# CELL H1 | Jalankan main.py sekali di dalam notebook dan TANGKAP input pipeline (tracking, ball, events, dst.)
import inspect
import os
import runpy
import sys
from pathlib import Path

import numpy as np
import pandas as pd

FPS = 25


def find_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "main.py").is_file():
            return p
    raise FileNotFoundError("main.py tidak ditemukan dari folder kerja notebook.")


ROOT = find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.possession_v2 as pv2

CAP = {}
_orig = pv2.run_controlled_possession
_sig = inspect.signature(_orig)


def _spy(*args, **kwargs):
    CAP.update(_sig.bind(*args, **kwargs).arguments)
    out = _orig(*args, **kwargs)
    CAP["labels"], CAP["diag"] = out
    return out


pv2.run_controlled_possession = _spy
_old_argv = sys.argv
sys.argv = ["main.py", "--config", "config.yaml", "--output", str(ROOT / "outputs" / "h1_tmp.csv")]
try:
    runpy.run_path(str(ROOT / "main.py"), run_name="__main__")
except SystemExit:
    pass
finally:
    sys.argv = _old_argv
    pv2.run_controlled_possession = _orig

print("tertangkap:", sorted(CAP))

Reading tactical tracking (this requires the full licensed ATD)...
Loaded 199,319 frames; 46 track slots
dead ball on 37.9% of graded frames (128 stoppages, 127 live stretches)
touch episodes: 1,345, controlled: 942
team-shape model: 5,909 pseudo-labelled frames; agreement with the detector on the held-out half: 86.8%, 85.6%
Wrote C:\Users\nesha\VSCODE_ALL\more-than-a-hack-mbg\outputs\h1_tmp.csv
Validated labels: {'X': 40462, 'D': 60222, 'A': 35420, 'H': 63215}
Run the organizer's evaluation/validate_submission.py on this exact file too.
tertangkap: ['ball', 'config', 'diag', 'events', 'halves', 'labels', 'tactical', 'tracking']


In [19]:
# CELL H2 | Lihat struktur objek yang ditangkap (tempel outputnya ke aku)
def brief(v):
    if isinstance(v, np.ndarray):
        return f"ndarray {v.shape} {v.dtype}"
    if isinstance(v, (list, tuple)):
        return f"{type(v).__name__}[{len(v)}]"
    if isinstance(v, dict):
        return f"dict[{len(v)}]"
    return f"{type(v).__name__} = {str(v)[:50]}"


for name in ("tracking", "ball", "tactical"):
    obj = CAP[name]
    print(f"== {name}: {type(obj).__name__}")
    for k, v in vars(obj).items():
        print(f"   {k:22} {brief(v)}")
print("== halves:", CAP["halves"])
print("== events: n =", len(CAP["events"]), "| contoh:", CAP["events"][:2])

== tracking: Tracking
   players_xy             ndarray (199319, 46, 2) float32
   ball_xy                ndarray (199319, 2) float32
   ball_speed             ndarray (199319,) float32
   teams                  ndarray (46,) int8
   fps                    int = 25
   pitch                  tuple[2]
== ball: BallEstimate
   xy                     ndarray (199319, 2) float32
   confidence             ndarray (199319,) float32
   observed               ndarray (199319,) bool
== tactical: TacticalEvidence
   log_odds_home          ndarray (199319,) float32
   set_piece              ndarray (199319,) float32
   event_count            int = 638
== halves: [(1, 11185, 90651), (2, 110808, 190197)]
== events: n = 638 | contoh: [{'code': 'SET PIECES', 'start': 11097, 'end': 11222, 'tags': {'team': 'N/A', 'type': 'Kick Off (Start)', 'half': '1st Half'}}, {'code': 'SET PIECES', 'start': 49631, 'end': 49756, 'tags': {'team': 'N/A', 'type': 'Kick Off (After Goal)', 'half': '1st Half'}}]


In [20]:
# CELL H3 | Di tiap pergantian tim di GT publik (H<->A langsung): kondisi sinyal dan kapan model berganti
from src.motion import compute_motion
from src.control import touch_owner

LO, HI = 11185, 18685
GT_PATH = ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"
gt = pd.read_csv(GT_PATH)["label"].astype(str).str.strip().str.upper().to_numpy()
pred = np.asarray(CAP["labels"])

cfg2 = CAP["config"]["possession_v2"]
motion = compute_motion(CAP["tracking"], CAP["ball"], CAP["halves"], cfg2["motion"])
owner = touch_owner(motion, cfg2["control"])   # 0 = H, 1 = A, -1 = tidak ada


def clean(a):
    a = np.asarray(a, dtype=float)
    return np.where(np.isfinite(a), a, np.nan)


near = {"H": clean(motion.nearest_h), "A": clean(motion.nearest_a)}
other = {"H": "A", "A": "H"}
team_idx = {"H": 0, "A": 1}
bspeed = clean(motion.ball_speed)

rows = []
for t in range(LO + 1, HI):
    a, b = gt[t - 1], gt[t]
    if a in ("H", "A") and b in ("H", "A") and a != b:
        w = slice(max(LO, t - FPS), min(HI, t + FPS + 1))
        nb, na = near[b], near[a]
        cand = [f for f in range(t - 5 * FPS, t + 10 * FPS) if pred[f] == b and pred[f - 1] != b]
        lag = min(cand, key=lambda f: abs(f - t)) - t if cand else None
        rows.append({
            "detik": round((t - LO) / FPS, 1),
            "GT": f"{a}->{b}",
            "dekat_tim_baru_m": None if np.isnan(nb[t]) else round(float(nb[t]), 1),
            "dekat_tim_baru_min_1s": None if np.all(np.isnan(nb[w])) else round(float(np.nanmin(nb[w])), 1),
            "dekat_tim_lama_m": None if np.isnan(na[t]) else round(float(na[t]), 1),
            "bola_m/s": None if np.isnan(bspeed[t]) else round(float(bspeed[t]), 1),
            "bola_terlihat_1s": round(float(motion.ball_known[w].mean()), 2),
            "sentuhan_tim_baru_1s": bool((owner[w] == team_idx[b]).any()),
            "model_lag_s": None if lag is None else round(lag / FPS, 2),
        })
pd.DataFrame(rows)

,detik,GT,dekat_tim_baru_m,dekat_tim_baru_min_1s,dekat_tim_lama_m,bola_m/s,bola_terlihat_1s,sentuhan_tim_baru_1s,model_lag_s
0,6.6,A->H,3.4,3.0,2.2,6.4,1.00,False,4.80
1,14.7,H->A,2.5,0.3,4.8,7.2,1.00,True,-1.48
2,17.6,A->H,1.1,0.7,4.0,0.7,1.00,True,-0.20
3,59.2,H->A,7.0,0.9,11.4,18.9,1.00,False,-0.16
4,100.8,H->A,1.8,0.4,3.6,10.6,1.00,True,NaN
5,104.0,A->H,7.9,6.7,7.5,0.8,1.00,False,NaN
6,146.4,H->A,1.1,0.2,3.8,6.7,1.00,True,-0.04
7,166.4,A->H,2.1,0.8,1.3,3.8,1.00,True,NaN
8,167.2,H->A,2.9,0.8,1.5,6.8,1.00,True,NaN
9,183.2,A->H,3.3,2.5,2.5,2.8,1.00,False,-4.60


In [21]:
# CELL H4 | Tiap pergantian H<->A di GT publik: episode sentuhan di sekitarnya, dikontrol atau tidak, dan label model
from src.live_dead import detect_dead_ball
from src.control import control_evidence

fps = CAP["tracking"].fps
blocks = [(first, last + 1) for _, first, last in CAP["halves"]]
graded = np.zeros(len(pred), dtype=bool)
for s_, e_ in blocks:
    graded[s_:e_] = True

dead, _ = detect_dead_ball(motion, CAP["events"], blocks, cfg2["dead_ball"], fps)
_, episodes = control_evidence(motion, CAP["ball"].xy, CAP["tracking"].pitch,
                               dead | ~graded, cfg2["control"], fps)

trans = [t for t in range(LO + 1, HI)
         if gt[t - 1] in ("H", "A") and gt[t] in ("H", "A") and gt[t - 1] != gt[t]]

rows = []
for t in trans:
    for e in episodes:
        if e.start < t + 2 * FPS and e.end > t - 2 * FPS:
            team = "HA"[e.team]
            mn = near[team][e.start:e.end]
            rows.append({
                "GT_detik": round((t - LO) / FPS, 1),
                "GT": f"{gt[t - 1]}->{gt[t]}",
                "ep_tim": team,
                "ep_mulai_s": round((e.start - LO) / FPS, 2),
                "ep_dur_s": round((e.end - e.start) / FPS, 2),
                "dikontrol": e.reason if e.controlled else "tidak",
                "min_jarak_m": None if np.all(np.isnan(mn)) else round(float(np.nanmin(mn)), 1),
                "model_di_ep": pred[(e.start + e.end) // 2],
            })
pd.DataFrame(rows)

,GT_detik,GT,ep_tim,ep_mulai_s,ep_dur_s,dikontrol,min_jarak_m,model_di_ep
0,6.6,A->H,A,6.32,0.04,tidak,1.5,A
1,14.7,H->A,H,11.44,1.48,kept,0.2,H
2,14.7,H->A,H,13.12,0.12,pass,0.6,H
3,14.7,H->A,A,15.56,0.20,tidak,0.3,A
4,14.7,H->A,H,15.84,0.20,tidak,0.9,A
5,14.7,H->A,H,16.36,0.40,tidak,0.6,A
6,17.6,A->H,A,15.56,0.20,tidak,0.3,A
7,17.6,A->H,H,15.84,0.20,tidak,0.9,A
8,17.6,A->H,H,16.36,0.40,tidak,0.6,A
9,17.6,A->H,H,17.36,2.92,kept,0.3,H


# Cell T - Cek kasus bola tidak terdeteksi sama sekali (tidak terdeteksi sentuhan)

In [19]:
# CELL I1 | Cari titik pergantian GT untuk kasus 6,6 s dan 183,2 s di window publik, lalu muat frame sekitarnya
import io
from pathlib import Path

import numpy as np
import pandas as pd

# Harus sudah ada dari CELL G1, G2, G3 (jalankan itu dulu di kernel yang sama)
_needed = ["ROOT", "FPS", "DATA", "paths", "parse_line", "SPOT", "GIF_DIR", "slot_team",
           "PITCH_L", "PITCH_W", "FLIP_Y", "fig", "ax", "title", "home_sc", "away_sc",
           "ball_sc", "trail", "xy_m", "hms"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise NameError(f"Belum ada: {_missing}. Jalankan CELL G1, G2, G3 dulu (G4 tidak perlu).")

LO, HI = 11185, 18685
GT_PATH = ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"
PRED_PATH = ROOT / "outputs" / "submission.csv"
TARGETS_S = [6.6, 183.2]     # detik dari awal window publik; ubah kalau mau kasus lain
PRE_S, POST_S = 10.0, 4.0     # tonton 4 detik sebelum dan sesudah titik pergantian GT
STEP_I = 2                   # ambil tiap 2 frame (0,08 s); ubah ke 1 untuk lebih halus
DURATION_I = 160             # ms per frame di GIF (sekitar 0,5x kecepatan asli)


def _read(p):
    return pd.read_csv(p)["label"].astype(str).str.strip().str.upper().to_numpy()


gt_lab, pred_lab = _read(GT_PATH), _read(PRED_PATH)
trans = [t for t in range(LO + 1, HI)
         if gt_lab[t - 1] in ("H", "A") and gt_lab[t] in ("H", "A") and gt_lab[t - 1] != gt_lab[t]]

cases = {}
for ts in TARGETS_S:
    t = min(trans, key=lambda x: abs((x - LO) / FPS - ts))
    s = max(0, t - int(PRE_S * FPS))
    e = min(199318, t + int(POST_S * FPS))
    cases[ts] = (t, s, e)
    print(f"kasus {ts} s: GT {gt_lab[t - 1]}->{gt_lab[t]} di frame {t} (video {hms(t)}), jendela {s}-{e}")

need_i = set()
for t, s, e in cases.values():
    need_i.update(range(s, e + 1))
need_i -= set(DATA)
max_i = max(need_i) if need_i else 0
with open(paths["atd_raw"], "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i in need_i:
            fr, arr, ball = parse_line(line)
            if fr != i + 1:
                raise ValueError(f"baris {i} berisi frame_id {fr}, harusnya {i + 1}")
            DATA[i] = (arr, ball)
        if i >= max_i:
            break
print(len(DATA), "frame di memori")

kasus 6.6 s: GT A->H di frame 11351 (video 0:07:34.04), jendela 11101-11451
kasus 183.2 s: GT A->H di frame 15766 (video 0:10:30.64), jendela 15516-15866
8335 frame di memori


In [20]:
# CELL I2 | Fungsi gambar untuk kasus ini: judul berisi label GT dan model, plus lingkaran radius 1,5 m di bola
import io

import numpy as np
from matplotlib.patches import Circle
from PIL import Image

if "cases" not in globals():
    raise NameError("Jalankan CELL I1 dulu.")

RING_R = 1.5   # sama dengan own_radius_m di config.yaml (dipakai model sebagai "dekat bola")
ring = Circle((-100, -100), RING_R, fill=False, ls="--", ec="black", lw=1.2, zorder=6)
ax.add_patch(ring)


def render_case(frame, t_gt):
    arr, ball = DATA[frame]
    pos = arr[:, :2]
    valid = np.isfinite(pos).all(axis=1)
    home_sc.set_offsets(xy_m(pos[valid & (slot_team == 0)]))
    away_sc.set_offsets(xy_m(pos[valid & (slot_team == 1)]))
    ball_ok = bool(np.isfinite(ball[:2]).all())
    if ball_ok:
        b = xy_m(ball[None, :2])
        ball_sc.set_offsets(b)
        ring.set_center((float(b[0, 0]), float(b[0, 1])))
    else:
        ball_sc.set_offsets(np.empty((0, 2)))
        ring.set_center((-100, -100))
    pts = [DATA[k][1][:2] for k in range(max(0, frame - 12), frame + 1)
           if k in DATA and np.isfinite(DATA[k][1][:2]).all()]
    if len(pts) >= 2:
        tr = xy_m(np.array(pts))
        trail.set_data(tr[:, 0], tr[:, 1])
    else:
        trail.set_data([], [])
    status = "" if ball_ok else "   BOLA TIDAK TERDETEKSI"
    title.set_text(f"frame {frame}   video {hms(frame)}   {(frame - t_gt) / FPS:+.2f}s dari pergantian GT\n"
                   f"GT: {gt_lab[frame]}     model: {pred_lab[frame]}{status}")
    title.set_color("black" if ball_ok else "#c0392b")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=70)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

In [21]:
# CELL I3 | Simpan GIF tiap kasus ke outputs/spotcheck/gif/ dan cetak urutan label GT vs model
from PIL import Image

if "render_case" not in globals():
    raise NameError("Jalankan CELL I2 dulu.")


def runs_txt(lab, s, e, t_gt):
    out, st = [], s
    for f in range(s + 1, e + 2):
        if f == e + 1 or lab[f] != lab[st]:
            out.append(f"{lab[st]} {(st - t_gt) / FPS:+.1f}..{(f - t_gt) / FPS:+.1f}s")
            st = f
    return " | ".join(out)


for ts, (t, s, e) in cases.items():
    imgs = [render_case(f, t).convert("P", palette=Image.ADAPTIVE, colors=64)
            for f in range(s, e + 1, STEP_I)]
    path = GIF_DIR / f"kasus_gt_{ts}s.gif"
    imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=DURATION_I,
                 loop=0, optimize=False, disposal=2)
    print(f"{path.name} | {len(imgs)} frame | pergantian GT di video {hms(t)}")
    print("   GT   :", runs_txt(gt_lab, s, e, t))
    print("   model:", runs_txt(pred_lab, s, e, t))

kasus_gt_6.6s.gif | 176 frame | pergantian GT di video 0:07:34.04
   GT   : X -10.0..-6.6s | A -6.6..+0.0s | H +0.0..+4.0s
   model: X -10.0..-6.6s | D -6.6..-6.6s | A -6.6..+4.0s
kasus_gt_183.2s.gif | 176 frame | pergantian GT di video 0:10:30.64
   GT   : A -10.0..+0.0s | H +0.0..+1.3s | A +1.3..+4.0s
   model: A -10.0..-4.6s | H -4.6..+4.0s
